# PART D & E: FULL RAG PIPELINE & CRITICAL EVALUATION
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  
**Course:** CS4241 - Introduction to Artificial Intelligence

---

---
## 🔵 PART D: FULL RAG PIPELINE IMPLEMENTATION
**Objective:** Build a complete pipeline: User Query → Retrieval → Context Selection → Prompt → LLM → Response, with logging at each stage.

### D.1 — Imports and Initialization

In [ ]:
import json
import faiss
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("✅ Client initialized.")

### D.2 — Load Chunks and Build Retrieval Index

In [ ]:
with open('chunks/csv_chunks.json', 'r') as f: csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f: pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks

model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('indexes/rag_index.faiss')

texts = [c['text'] for c in all_chunks]
bm25 = BM25Okapi([t.lower().split() for t in texts])
print(f"✅ Loaded {len(all_chunks)} total chunks.")

### D.3 — The Full RAG Pipeline Function (with staged logging)

In [ ]:
def rag_pipeline(query, k=3, verbose=True):
    if verbose: print(f"[LOG] Starting pipeline for query: '{query}'")
    
    # Stage 1: Retrieval
    if verbose: print("[LOG] Stage 1: Retrieving relevant documents...")
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1

    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i: v_score = 1 / (1 + distances[0][rank])
        score = (0.5 * v_score) + (0.5 * (bm25_scores[i] / max_bm25))
        if score > 0: combined.append({'chunk': chunk, 'score': score})

    results = sorted(combined, key=lambda x: x['score'], reverse=True)[:k]
    
    if verbose:
        print(f"[LOG] Found {len(results)} relevant chunks.")
        for i, res in enumerate(results):
            print(f"      - Chunk {i+1} | Score: {res['score']:.4f} | Source: {res['chunk']['source']}")

    # Stage 2: Context Selection
    if verbose: print("[LOG] Stage 2: Building context block...")
    context = "".join([f"\n[Source: {r['chunk']['source']}] {r['chunk']['text']}\n" for r in results])

    # Stage 3: Prompt Construction
    if verbose: print("[LOG] Stage 3: Constructing final prompt...")
    final_prompt = f"""Answer the question based only on the provided documents.
If the answer is not in the context, say you do not know.

Documents:
{context}

Question: {query}
Answer:"""

    if verbose:
        print("--- FINAL PROMPT SENT TO LLM ---")
        print(final_prompt[:600] + "...")
        print("--- END OF PROMPT ---")

    # Stage 4: LLM Generation
    if verbose: print("[LOG] Stage 4: Querying LLM...")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    return response.choices[0].message.content

print("✅ Pipeline function defined.")

### D.4 — Part D Test: Normal Query (Pipeline Verification)

In [ ]:
normal_query = "What was the total expenditure in 2024?"
answer = rag_pipeline(normal_query)
print(f"\n--- FINAL RESPONSE ---\n{answer}")

---
---
## 🔴 PART E: CRITICAL EVALUATION & ADVERSARIAL TESTING
**Objective:** Design 2 adversarial queries, run them against both Pure LLM and RAG, then evaluate on Accuracy, Hallucination Rate, and Response Consistency.

### E.1 — Pure LLM Helper Function (No Retrieval)

In [ ]:
def pure_llm_response(query):
    """Calls the LLM with NO context — simulates a pure chatbot with no RAG."""
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    return res.choices[0].message.content

print("✅ Pure LLM function defined.")

### E.2 — Adversarial Test 1: Ambiguous Query
> **Query:** `Who won the election?`  
> **Why it's adversarial:** No year, no region, no country is specified. An unreliable system would guess.

In [ ]:
adversarial_q1 = "Who won the election?"
print(f"{'='*50}")
print(f"TEST TYPE: Ambiguous")
print(f"QUERY: {adversarial_q1}")
print(f"{'='*50}\n")

print("--- PURE LLM RESPONSE ---")
pure_q1 = pure_llm_response(adversarial_q1)
print(pure_q1)

print("\n--- RAG SYSTEM RESPONSE ---")
rag_q1 = rag_pipeline(adversarial_q1, verbose=False)
print(rag_q1)

### E.3 — Evaluation of Test 1 (Ambiguous Query)

In [ ]:
print("EVALUATION — Test 1: Ambiguous Query")
print("=" * 50)

print("\n[1] ACCURACY")
print("  Pure LLM : FAIL — Could not answer. Asked for more context.")
print("  RAG      : PARTIAL PASS — Could not confirm winner from docs, but correctly")
print("             cited 'H.E John Mahama's administration' from the budget text.")
print("             This is grounded and accurate to the source material.")

print("\n[2] HALLUCINATION RATE")
print("  Pure LLM : LOW — Did not hallucinate; admitted ignorance.")
print("  RAG      : ZERO — Refused to guess; cited source document explicitly.")

print("\n[3] RESPONSE CONSISTENCY")
print("  Pure LLM : VARIABLE — May give different answers depending on phrasing.")
print("  RAG      : CONSISTENT — Tethered to fixed index; same answer every run.")

### E.4 — Adversarial Test 2: Misleading Query
> **Query:** `How many votes did the NPP get in the 2025 budget?`  
> **Why it's adversarial:** It deliberately mixes election data (votes) with budget data. A hallucinating system would blend both.

In [ ]:
adversarial_q2 = "How many votes did the NPP get in the 2025 budget?"
print(f"{'='*50}")
print(f"TEST TYPE: Misleading")
print(f"QUERY: {adversarial_q2}")
print(f"{'='*50}\n")

print("--- PURE LLM RESPONSE ---")
pure_q2 = pure_llm_response(adversarial_q2)
print(pure_q2)

print("\n--- RAG SYSTEM RESPONSE ---")
rag_q2 = rag_pipeline(adversarial_q2, verbose=False)
print(rag_q2)

### E.5 — Evaluation of Test 2 (Misleading Query)

In [ ]:
print("EVALUATION — Test 2: Misleading Query")
print("=" * 50)

print("\n[1] ACCURACY")
print("  Pure LLM : PASS — Correctly noted its knowledge cutoff and declined to answer.")
print("  RAG      : PASS — Said 'I don't know'. Correctly refused to mix election")
print("             results with budget documents.")

print("\n[2] HALLUCINATION RATE")
print("  Pure LLM : LOW — Did not hallucinate; cited its knowledge cutoff.")
print("  RAG      : ZERO — Did not invent vote counts from unrelated documents.")

print("\n[3] RESPONSE CONSISTENCY")
print("  Pure LLM : MODERATE — A different LLM model might attempt an answer.")
print("  RAG      : CONSISTENT — No budget document contains vote counts, so the")
print("             answer will always be 'I don't know' regardless of how often asked.")

### E.6 — Final Summary Comparison Table

In [ ]:
import pandas as pd

summary = {
    "Query": [
        "Who won the election? (Ambiguous)",
        "NPP votes in 2025 budget? (Misleading)"
    ],
    "Accuracy — Pure LLM": ["❌ Failed", "✅ Passed"],
    "Accuracy — RAG": ["✅ Partial (cited source)", "✅ Passed"],
    "Hallucination — Pure LLM": ["Low (admitted ignorance)", "Low (cited cutoff)"],
    "Hallucination — RAG": ["ZERO", "ZERO"],
    "Consistency — Pure LLM": ["Variable", "Moderate"],
    "Consistency — RAG": ["High", "High"]
}

df = pd.DataFrame(summary)
print(df.to_string(index=False))